### Phương pháp Xuống thang


In [4]:
import numpy as np
from IPython.display import display, Math, Markdown

def _mat(M, p=4):
    """Chuyển đổi ma trận hoặc mảng NumPy thành chuỗi LaTeX bmatrix để hiển thị."""
    if hasattr(M[0], "__len__"):
        rows = " \\\\ ".join([" & ".join([f"{x:.{p}f}" for x in row]) for row in M])
        return f"\\begin{{bmatrix}} {rows} \\end{{bmatrix}}"
    inner = " \\\\ ".join([f"{x:.{p}f}" for x in M])
    return f"\\begin{{bmatrix}} {inner} \\end{{bmatrix}}^T"

def Power_Method_Step(M, x0, tol=1e-6, max_iter=300):
    """Thuật toán lũy thừa tìm trị riêng trội và véctơ riêng chuẩn hóa chuẩn 2."""
    x = x0 / np.linalg.norm(x0, 2)
    lam_prev = 0
    for k in range(max_iter):
        y = M @ x
        lam = np.dot(x, y)
        if np.linalg.norm(y, 2) == 0:
            return 0.0, x
        x_new = y / np.linalg.norm(y, 2)
        if abs(lam - lam_prev) < tol:
            return lam, x_new
        lam_prev = lam
        x = x_new
    return lam, x

def Tim_Gia_Tri_Ky_Di_Deflation(A_input, y0_input):
    A = np.array(A_input, dtype=float)
    y0 = np.array(y0_input, dtype=float).flatten()
    
    display(Markdown("# BÁO CÁO TOÁN HỌC: TÌM TRỊ RIÊNG VÀ GIÁ TRỊ KỲ DỊ QUA XUỐNG THANG"))
    display(Markdown("---"))
    
    # -----------------------------------------------------------------
    # GIAI ĐOẠN 1: KHỞI TẠO HỆ THỐNG
    # -----------------------------------------------------------------
    display(Markdown("### I. Khởi tạo và Cơ sở lý thuyết"))
    display(Markdown(f"Ma trận gốc $A_{{4 \\times 3}}$ và véctơ khởi đầu $y$ :"))
    display(Math(f"A = {_mat(A)} \\quad ; \\quad y = {_mat(y0)}"))
    
    # Vì y thuộc R^4, ta xét ma trận đối xứng dòng A * A^T
    M = A @ A.T
    display(Markdown("Do véctơ khởi đầu $y \\in \\mathbb{R}^4$, ta xây dựng ma trận đối xứng $M = A \\cdot A^T \\in \\mathbb{R}^{4 \\times 4}$:"))
    display(Math(f"M = A \\cdot A^T = {_mat(M)}"))
    display(Markdown("> **Nguyên lý:** Trị riêng của $M$ là $\\lambda_i$. Giá trị kỳ dị của $A$ là $\\sigma_i = \\sqrt{\\lambda_i}$."))
    
    # -----------------------------------------------------------------
    # GIAI ĐOẠN 2: LẶP XUỐNG THANG TỪNG BƯỚC
    # -----------------------------------------------------------------
    display(Markdown("\n---\n### II. Quá trình tính toán chi tiết & Hạ bậc ma trận"))
    
    Current_M = np.copy(M)
    dst_lam = []
    dst_sigma = []
    dst_v = []
    
    # Ma trận kích thước 4x4, hạng tối đa là 3 (do từ A_4x3), ta tìm 3 thành phần trị riêng chính
    for step in range(1, 4):
        display(Markdown(f"#### **❑ BƯỚC XUỐNG THANG THỨ {step}**"))
        display(Markdown(f"Ma trận dùng ở bước này $M_{step}$ :"))
        display(Math(f"M_{step} = {_mat(Current_M)}"))
        
        # Tìm trị riêng và vtr trội bằng pp lũy thừa
        lam, v = Power_Method_Step(Current_M, y0)
        sigma = np.sqrt(max(0, lam))
        
        dst_lam.append(lam)
        dst_sigma.append(sigma)
        dst_v.append(v)
        
        display(Markdown(f"* **Véctơ riêng dùng để xuống thang (chuẩn 2 bằng 1):**"))
        display(Math(f"v_{step} = {_mat(v)}"))
        display(Markdown(f"* **Kết quả:** Trị riêng $\\lambda_{step} = {lam:.5f} \\Rightarrow$ Giá trị kỳ dị $\\sigma_{step} = {sigma:.5f}$"))
        
        # Cấu trúc ma trận hạ bậc Hotelling
        v_outer = np.outer(v, v)
        M_correction = lam * v_outer
        Current_M = Current_M - M_correction
        
        display(Markdown(f"* **Ma trận tích ngoài $v_{step} v_{step}^T$:**"))
        display(Math(f"v_{step} v_{step}^T = {_mat(v_outer)}"))
        display(Markdown(f"* **Ma trận cấu trúc khử trội ($\\lambda_{step} v_{step} v_{step}^T$):**"))
        display(Math(f"\\lambda_{step} v_{step} v_{step}^T = {_mat(M_correction)}"))
        display(Markdown("---"))

    # -----------------------------------------------------------------
    # GIAI ĐOẠN 3: TỔNG HỢP KẾT QUẢ THEO BẢNG HÀNG NGANG
    # -----------------------------------------------------------------
    display(Markdown("### III. Tổng hợp kết quả nghiệm toàn cục"))
    
    # Xây dựng bảng hiển thị kết quả trực quan
    header = "| Chỉ số ($i$) | Trị riêng $\\lambda_i$ (của $AA^T$) | Giá trị kỳ dị $\\sigma_i$ (của $A$) | Véctơ riêng gốc $v_i^T$ (chuẩn 2) |"
    sep = "|:---:|:---:|:---:|:---|"
    lines = [header, sep]
    
    for idx in range(3):
        v_str = ", ".join([f"{x:.5f}" for x in dst_v[idx]])
        row = f"| **{idx+1}** | {dst_lam[idx]:.5f} | **{dst_sigma[idx]:.5f}** | `[{v_str}]` |"
        lines.append(row)
        
    display(Markdown("\n".join(lines)))

# ═══════════════════════════════════════════════════════════════════
# KHU VỰC NHẬP DỮ LIỆU CHÍNH XÁC TỪ ĐỀ BÀI TRONG ẢNH
# ═══════════════════════════════════════════════════════════════════
A = np.array([
    [3,  3,  7],
    [3,  9, 10],
    [8,  2, 10],
    [2, 10,  9]
], dtype=float)

y0 = np.array([1, 1, 1, 1], dtype=float)
# ═══════════════════════════════════════════════════════════════════

# Thực thi chương trình để xuất báo cáo
Tim_Gia_Tri_Ky_Di_Deflation(A, y0)

# BÁO CÁO TOÁN HỌC: TÌM TRỊ RIÊNG VÀ GIÁ TRỊ KỲ DỊ QUA XUỐNG THANG

---

### I. Khởi tạo và Cơ sở lý thuyết

Ma trận gốc $A_{4 \times 3}$ và véctơ khởi đầu $y$ :

<IPython.core.display.Math object>

...

* **Ma trận cấu trúc khử trội ($\lambda_3 v_3 v_3^T$):**

<IPython.core.display.Math object>

---

### III. Tổng hợp kết quả nghiệm toàn cục

| Chỉ số ($i$) | Trị riêng $\lambda_i$ (của $AA^T$) | Giá trị kỳ dị $\sigma_i$ (của $A$) | Véctơ riêng gốc $v_i^T$ (chuẩn 2) |
|:---:|:---:|:---:|:---|
| **1** | 550.39869 | **23.46058** | `[0.34189, 0.57969, 0.48785, 0.55594]` |
| **2** | 58.68943 | **7.66090** | `[0.18247, -0.29301, 0.79277, -0.50237]` |
| **3** | 0.91188 | **0.95492** | `[0.88203, 0.06609, -0.36373, -0.29216]` |